# Rank sweep significance analysis

This notebook checks which configured rank values perform best and whether the differences are significant.

Here, configured rank means the native LoRA bottleneck rank used by the method. It does not normalize for number of experts, heads, up-projections, or CoLA A/B factors.


In [1]:
# first we load the data.
# All data is provided in a single excel report sheet which is the single source of trust here
from pathlib import Path
import pandas as pd

ROOT = Path("/local/joeldag/repos/moe-study")
REPORT = ROOT / "outputs/pipeline_runs/moe_study_rank_sweep_pipeline_report/rank_pipeline_report.xlsx"

xls = pd.ExcelFile(REPORT)
raw = pd.read_excel(REPORT, sheet_name="benchmark_raw")
source = pd.read_excel(REPORT, sheet_name="source_manifest")
audit = pd.read_excel(REPORT, sheet_name="audit_status")

primary = raw[
  ((raw.benchmark == "Belebele") & (raw.metric == "acc_norm")) |
  ((raw.benchmark == "FLORES") & (raw.metric.astype(str).str.startswith("bits_per_byte "))) |
  ((raw.benchmark == "SIB200") & (raw.metric == "acc")) |
  ((raw.benchmark == "Taxi1500") & (raw.metric == "acc"))
].copy() # primary metrics we use in our analysis

primary = primary[~primary["is_baseline"].astype(str).str.lower().isin(["true", "1", "yes"])] # remove baseline
primary.head()


,run_id,kind,benchmark,task,language,resource_availability,metric,better,value,is_baseline
1,cola_r128,cola,Belebele,belebele_arb_Arab,arb_Arab,HIGH,acc_norm,higher,0.228889,False
2,cola_r16,cola,Belebele,belebele_arb_Arab,arb_Arab,HIGH,acc_norm,higher,0.262222,False
3,cola_r2,cola,Belebele,belebele_arb_Arab,arb_Arab,HIGH,acc_norm,higher,0.407778,False
4,cola_r256,cola,Belebele,belebele_arb_Arab,arb_Arab,HIGH,acc_norm,higher,0.272222,False
5,cola_r32,cola,Belebele,belebele_arb_Arab,arb_Arab,HIGH,acc_norm,higher,0.218889,False


In [2]:
# here we prepare our df for analysis and normalize the scores
import re
import numpy as np

# helper, extracts the configured/native rank from run_id.
# For regular rank runs, e.g. lora_r16, hydralora_r16, mtllora_r16, cola_r16 -> 16
# For MoE rank runs, e.g. moelora_tr16_e8_k2 -> per-expert configured rank 16 / 8 = 2
def parse_configured_rank(run_id):
  run_id = str(run_id)
  total = re.search(r"_tr(\d+)_", run_id)
  experts = re.search(r"_e(\d+)(?:_|$)", run_id)
  if total and experts:
      return int(total.group(1)) / int(experts.group(1))

  rank = re.search(r"_r(\d+)(?:_|$)", run_id)
  if rank:
      return int(rank.group(1))
  return np.nan

# Keep the encoded total rank as annotation only; we do not use it as the main analysis axis.
def parse_encoded_total_rank(run_id):
  run_id = str(run_id)
  total = re.search(r"_tr(\d+)_", run_id)
  if total:
      return int(total.group(1))
  rank = re.search(r"_r(\d+)(?:_|$)", run_id)
  if rank:
      return int(rank.group(1))
  return np.nan

metrics = primary.copy()
metrics["direction"] = metrics.metric.astype(str).str.extract(r"(en->x|x->en)", expand=False).fillna("") # extract FLORES direction
metrics["score_family"] = np.where(metrics.benchmark.eq("FLORES"), "FLORES " + metrics.direction, metrics.benchmark)
metrics["configured_rank"] = metrics.run_id.map(parse_configured_rank)
metrics["encoded_total_rank"] = metrics.run_id.map(parse_encoded_total_rank)
metrics["value"] = pd.to_numeric(metrics.value, errors="coerce") # ensure numeric values, NaN else

# we normalize metrics using min-max normalization within each benchmark/task/metric.
# Higher is always better after normalization; lower-is-better metrics are inverted.
g = metrics.groupby(["score_family", "task", "metric"]).value
lo, hi = g.transform("min"), g.transform("max")
metrics["normalized_score"] = np.where(metrics.better.eq("lower"), (hi - metrics.value) / (hi - lo), (metrics.value - lo) / (hi - lo))
metrics.loc[(hi - lo).eq(0), "normalized_score"] = 1.0 # edge case if all values are the same

metrics = metrics.dropna(subset=["value", "configured_rank"]).copy()
metrics["configured_rank"] = metrics.configured_rank.astype(int)
metrics.head()


,run_id,kind,benchmark,task,language,resource_availability,metric,better,value,is_baseline,direction,score_family,configured_rank,encoded_total_rank,normalized_score
1,cola_r128,cola,Belebele,belebele_arb_Arab,arb_Arab,HIGH,acc_norm,higher,0.228889,False,,Belebele,128,128.0,0.025140
2,cola_r16,cola,Belebele,belebele_arb_Arab,arb_Arab,HIGH,acc_norm,higher,0.262222,False,,Belebele,16,16.0,0.108939
3,cola_r2,cola,Belebele,belebele_arb_Arab,arb_Arab,HIGH,acc_norm,higher,0.407778,False,,Belebele,2,2.0,0.474860
4,cola_r256,cola,Belebele,belebele_arb_Arab,arb_Arab,HIGH,acc_norm,higher,0.272222,False,,Belebele,256,256.0,0.134078
5,cola_r32,cola,Belebele,belebele_arb_Arab,arb_Arab,HIGH,acc_norm,higher,0.218889,False,,Belebele,32,32.0,0.000000


In [3]:
# sanity checks
print("rows:", len(metrics))
print("runs:", metrics.run_id.nunique())
print("configured rank values:", sorted(metrics.configured_rank.unique()))
print("score families:", metrics.score_family.unique())
print(metrics[["value", "normalized_score", "configured_rank"]].isna().sum())

# quick check of rank semantics for selected runs
semantics = (
  metrics[["run_id", "kind", "configured_rank", "encoded_total_rank"]]
  .drop_duplicates()
  .sort_values(["kind", "configured_rank", "run_id"])
)
display(semantics.head(20))


rows: 2268
runs: 54
configured rank values: [np.int64(2), np.int64(4), np.int64(8), np.int64(16), np.int64(32), np.int64(64), np.int64(128), np.int64(256), np.int64(512), np.int64(1024)]
score families: ['Belebele' 'FLORES en->x' 'FLORES x->en' 'SIB200' 'Taxi1500']
value               0
normalized_score    0
configured_rank     0
dtype: int64


,run_id,kind,configured_rank,encoded_total_rank
3,cola_r2,cola,2,2.0
6,cola_r4,cola,4,4.0
8,cola_r8,cola,8,8.0
2,cola_r16,cola,16,16.0
5,cola_r32,cola,32,32.0
7,cola_r64,cola,64,64.0
1,cola_r128,cola,128,128.0
4,cola_r256,cola,256,256.0
11,hmora_tr16_e8_k2,hmora,2,16.0
13,hmora_tr32_e8_k2,hmora,4,32.0


# Which configured rank value wins overall and per benchmark family?

This analysis uses configured rank, not total/stored capacity rank.
For example, `hydralora_r2`, `mtllora_r2`, and `cola_r2` are rank `2`; `moelora_tr16_e8_k2` is also configured rank `2` because it has total encoded rank 16 over 8 experts.


In [4]:
from scipy.stats import friedmanchisquare, wilcoxon
from statsmodels.stats.multitest import multipletests
import pandas as pd
import numpy as np

# We use Friedman first for broad comparison across configured rank values.
# Then we use Wilcoxon signed-rank tests between the top-ranked values.
# Holm correction is used for conservative post-hoc claims while controlling family-wise error rate.
# Reference: https://en.wikipedia.org/wiki/Holm%E2%80%93Bonferroni_method

def holm(pvals):
  return multipletests(pd.Series(pvals, dtype=float), method="holm")[1]


def rank_pivot(df, group_cols):
  return (
      df.groupby(group_cols + ["configured_rank"])["normalized_score"]
      .mean()
      .unstack("configured_rank")
      .dropna()
  )


def rank_friedman_overall(df, label="overall"):
  # Descriptive overview: average normalized score per configured rank value.
  rank_summary = (
      df.groupby("configured_rank")
      .agg(
          mean_score=("normalized_score", "mean"),
          n_rows=("normalized_score", "count"),
          n_approaches=("kind", "nunique"),
          approaches=("kind", lambda s: ", ".join(sorted(s.unique()))),
      )
      .sort_values("mean_score", ascending=False)
  )

  # Paired table for Friedman: rows are benchmark/task/language/metric units, columns are configured rank values.
  pivot = rank_pivot(df, ["score_family", "task", "language", "metric"])
  ranks = list(pivot.columns)

  stat, p = friedmanchisquare(*[pivot[r] for r in ranks])
  avg_ranks = pivot.rank(axis=1, ascending=False).mean().sort_values() # lower average rank is better
  top3 = avg_ranks.index[:3].tolist()

  return rank_summary, {
      "analysis": label,
      "pivot": pivot,
      "top3_ranks": top3,
      "friedman_p_raw": p,
      "kendalls_w": stat / (len(pivot) * (pivot.shape[1] - 1)),
      **{f"avg_rank_{r}": avg_ranks.get(r, np.nan) for r in ranks},
  }


def wilcoxon_top3(result):
  """
  Compare the Friedman-ranked top 3 configured rank values:
  1st vs 2nd, 1st vs 3rd, and 2nd vs 3rd.
  We use one-sided Wilcoxon signed-rank tests because the question is directional:
  whether the higher-ranked rank value performs better than the lower-ranked one.
  """
  pivot = result["pivot"]
  top3 = result["top3_ranks"]

  rows = []
  for a, b in [(top3[0], top3[1]), (top3[0], top3[2]), (top3[1], top3[2])]:
      w = wilcoxon(pivot[a], pivot[b], zero_method="pratt", alternative="greater")
      rows.append({
          "analysis": result["analysis"],
          "a": a,
          "b": b,
          "mean_a": pivot[a].mean(),
          "mean_b": pivot[b].mean(),
          "median_delta_a_minus_b": (pivot[a] - pivot[b]).median(),
          "n_pairs": len(pivot),
          "p_raw": w.pvalue,
      })

  out = pd.DataFrame(rows)
  out["p_holm"] = holm(out["p_raw"])
  out["significant"] = out["p_holm"] < 0.05
  return out


def rank_friedman_by_benchmark(df, label="overall"):
  rows = []

  for score_family, g in df.groupby("score_family"):
      pivot = rank_pivot(g, ["task", "language", "metric"])
      ranks = list(pivot.columns)

      stat, p = friedmanchisquare(*[pivot[r] for r in ranks])
      avg_ranks = pivot.rank(axis=1, ascending=False).mean().sort_values()
      top3 = avg_ranks.index[:3].tolist()

      rows.append({
          "analysis": label,
          "score_family": score_family,
          "winner_rank": avg_ranks.index[0],
          "top3_ranks": top3,
          "n_pairs": len(pivot),
          "p_raw": p,
          "kendalls_w": stat / (len(pivot) * (pivot.shape[1] - 1)),
          **{f"avg_rank_{r}": avg_ranks.get(r, np.nan) for r in ranks},
      })

  out = pd.DataFrame(rows)
  out["p_holm"] = holm(out["p_raw"])
  out["significant"] = out["p_holm"] < 0.05
  return out


# With CoLA
rank_summary_all, overall_all = rank_friedman_overall(metrics, "with_cola")
overall_test_all = pd.DataFrame([{k: v for k, v in overall_all.items() if k != "pivot"}])
overall_test_all["p_holm"] = holm(overall_test_all["friedman_p_raw"])
overall_test_all["significant"] = overall_test_all["p_holm"] < 0.05
wilcoxon_all = wilcoxon_top3(overall_all)
benchmark_all = rank_friedman_by_benchmark(metrics, "with_cola")

print("With CoLA: configured ranks sorted by mean normalized score")
display(rank_summary_all.head(10))
print("With CoLA: Friedman result ordered by average rank")
display(overall_test_all)
print("With CoLA: Wilcoxon tests between Friedman top 3 configured ranks")
display(wilcoxon_all)
print("With CoLA: best configured rank per benchmark family")
display(benchmark_all)

# Without CoLA
no_cola_metrics = metrics[metrics.kind.ne("cola")].copy()
rank_summary_no_cola, overall_no_cola = rank_friedman_overall(no_cola_metrics, "no_cola")
overall_test_no_cola = pd.DataFrame([{k: v for k, v in overall_no_cola.items() if k != "pivot"}])
overall_test_no_cola["p_holm"] = holm(overall_test_no_cola["friedman_p_raw"])
overall_test_no_cola["significant"] = overall_test_no_cola["p_holm"] < 0.05
wilcoxon_no_cola = wilcoxon_top3(overall_no_cola)
benchmark_no_cola = rank_friedman_by_benchmark(no_cola_metrics, "no_cola")

print("Without CoLA: configured ranks sorted by mean normalized score")
display(rank_summary_no_cola.head(10))
print("Without CoLA: Friedman result ordered by average rank")
display(overall_test_no_cola)
print("Without CoLA: Wilcoxon tests between Friedman top 3 configured ranks")
display(wilcoxon_no_cola)
print("Without CoLA: best configured rank per benchmark family")
display(benchmark_no_cola)


With CoLA: configured ranks sorted by mean normalized score


,mean_score,n_rows,n_approaches,approaches
configured_rank,,,,
2,0.899756,294,7,"cola, hmora, hydralora, lora, moelora, mola, m..."
4,0.883531,294,7,"cola, hmora, hydralora, lora, moelora, mola, m..."
8,0.826811,294,7,"cola, hmora, hydralora, lora, moelora, mola, m..."
1024,0.817166,42,1,lora
512,0.808041,42,1,lora
16,0.791721,294,7,"cola, hmora, hydralora, lora, moelora, mola, m..."
32,0.754383,294,7,"cola, hmora, hydralora, lora, moelora, mola, m..."
64,0.734177,294,7,"cola, hmora, hydralora, lora, moelora, mola, m..."
128,0.629969,252,6,"cola, hmora, hydralora, lora, mola, mtllora"


With CoLA: Friedman result ordered by average rank


,analysis,top3_ranks,friedman_p_raw,kendalls_w,avg_rank_2,avg_rank_4,avg_rank_8,avg_rank_16,avg_rank_32,avg_rank_64,avg_rank_128,avg_rank_256,avg_rank_512,avg_rank_1024,p_holm,significant
0,with_cola,"[2, 4, 1024]",1.545907e-60,0.808957,1.321429,2.25,4.357143,5.404762,6.857143,7.261905,8.738095,9.928571,4.666667,4.214286,1.545907e-60,True


With CoLA: Wilcoxon tests between Friedman top 3 configured ranks


,analysis,a,b,mean_a,mean_b,median_delta_a_minus_b,n_pairs,p_raw,p_holm,significant
0,with_cola,2,4,0.899756,0.883531,0.002588,42,1.648016e-06,0.000003,True
1,with_cola,2,1024,0.899756,0.817166,0.020107,42,8.947236e-07,0.000003,True
2,with_cola,4,1024,0.883531,0.817166,0.009331,42,3.450008e-05,0.000035,True


... (6 outputs left out)

Without CoLA: Wilcoxon tests between Friedman top 3 configured ranks


,analysis,a,b,mean_a,mean_b,median_delta_a_minus_b,n_pairs,p_raw,p_holm,significant
0,no_cola,2,4,0.927846,0.917669,0.000227,42,0.005570,0.011140,True
1,no_cola,2,8,0.927846,0.915726,0.001222,42,0.000006,0.000019,True
2,no_cola,4,8,0.917669,0.915726,0.000177,42,0.121309,0.121309,False


Without CoLA: best configured rank per benchmark family


,analysis,score_family,winner_rank,top3_ranks,n_pairs,p_raw,kendalls_w,avg_rank_2,avg_rank_4,avg_rank_8,avg_rank_16,avg_rank_32,avg_rank_64,avg_rank_128,avg_rank_256,avg_rank_512,avg_rank_1024,p_holm,significant
0,no_cola,Belebele,2,"[2, 8, 4]",9,4.807695e-11,0.832697,1.777778,2.555556,2.000000,4.333333,5.555556,6.444444,8.000000,8.666667,9.333333,6.333333,9.615391e-11,True
1,no_cola,FLORES en->x,2,"[2, 4, 8]",9,1.593886e-13,0.988028,1.000000,2.000000,3.000000,4.000000,5.000000,7.777778,8.888889,10.000000,7.222222,6.111111,6.375543e-13,True
2,no_cola,FLORES x->en,2,"[2, 4, 8]",9,1.221360e-13,0.995211,1.000000,2.111111,2.888889,4.000000,5.000000,7.888889,9.000000,10.000000,7.111111,6.000000,6.106800e-13,True
3,no_cola,SIB200,4,"[4, 2, 8]",9,1.461457e-11,0.865320,2.333333,2.222222,3.444444,3.444444,6.222222,6.444444,8.222222,9.888889,3.888889,8.888889,4.384370e-11,True
4,no_cola,Taxi1500,2,"[2, 1024, 4]",6,1.172979e-06,0.822885,1.666667,3.250000,3.333333,4.916667,6.166667,8.666667,8.833333,9.500000,5.500000,3.166667,1.172979e-06,True


### Interpretation
- This analysis uses configured rank, not total/stored capacity rank.
- Therefore `hydralora_r2`, `mtllora_r2`, `cola_r2`, and `lora_r2` are all treated as configured rank `2`; MoE runs such as `moelora_tr16_e8_k2` are also configured rank `2` because `16 / 8 = 2`.
- This answers which native rank setting works best on average, but it does not normalize for parameter capacity, number of experts, heads, up-projections, or CoLA factor combinations


 ### results
based on the configured-rank analysis, rank `2` performs best and rank `4` is second best. The difference between rank `2` and rank `4` is statistically significant according to the Holm-corrected Wilcoxon signed-rank test.
The difference between the second best rank `4` and third best `8` is not statistically significant.

This suggests that lower configured ranks work better in this sweep. One possible explanation is the limited training budget, where larger-rank configurations may need more training to converge properly


# Which concrete approaches work best at the best configured ranks?

After finding the best configured rank values, we now inspect the actual run configurations inside the top two ranks. This answers which concrete approach performs best when configured with the strongest rank settings


In [9]:
def concrete_config_pivot(df, group_cols):
  return (
      df.groupby(group_cols + ["run_id"])["normalized_score"]
      .mean()
      .unstack("run_id")
      .dropna()
  )


def best_configs_with_cola(df, configured_rank, top_config_n=4):
  selected = df[df.configured_rank.eq(configured_rank)].copy()

  # Descriptive ranking of concrete run configs inside the selected configured rank.
  config_summary = (
      selected.groupby(["configured_rank", "run_id", "kind"])
      .agg(
          mean_score=("normalized_score", "mean"),
          n_rows=("normalized_score", "count"),
      )
      .sort_values("mean_score", ascending=False)
      .reset_index()
  )

  candidates = config_summary.head(top_config_n)["run_id"].tolist()
  pivot = concrete_config_pivot(
      selected[selected.run_id.isin(candidates)],
      ["score_family", "task", "language", "metric"],
  )

  # Friedman tests whether the top concrete configs differ overall.
  stat, p = friedmanchisquare(*[pivot[c] for c in candidates])
  avg_ranks = pivot.rank(axis=1, ascending=False).mean().sort_values()
  ordered = avg_ranks.index.tolist()

  friedman_result = pd.DataFrame([{
      "configured_rank": configured_rank,
      "top_configs_by_friedman_rank": ordered,
      "n_pairs": len(pivot),
      "friedman_p_raw": p,
      "friedman_p_holm": holm([p])[0],
      "significant": holm([p])[0] < 0.05,
      "kendalls_w": stat / (len(pivot) * (pivot.shape[1] - 1)),
      **{f"avg_rank_{c}": avg_ranks.get(c, np.nan) for c in ordered},
  }])

  # Post-hoc: compare Friedman top1 vs 2nd, 3rd, and 4th using Wilcoxon + Holm.
  rows = []
  best = ordered[0]
  for other in ordered[1:]:
      w = wilcoxon(pivot[best], pivot[other], zero_method="pratt", alternative="greater")
      rows.append({
          "a": best,
          "b": other,
          "mean_a": pivot[best].mean(),
          "mean_b": pivot[other].mean(),
          "median_delta_a_minus_b": (pivot[best] - pivot[other]).median(),
          "n_pairs": len(pivot),
          "p_raw": w.pvalue,
      })

  pairwise = pd.DataFrame(rows)
  pairwise["p_holm"] = holm(pairwise["p_raw"])
  pairwise["significant"] = pairwise["p_holm"] < 0.05
  return config_summary, friedman_result, pairwise


print("These are the best concrete configs within the top 2 configured ranks")
top_ranks_all = overall_all["top3_ranks"][:2]
print("Top configured ranks:", top_ranks_all)

print("\n\n now we check within the best two ranks overll, which are specifically the best approaches within those")
for rank in top_ranks_all:
  print(f"Configured rank {rank}: best concrete configs")
  config_summary_all, config_friedman_all, config_pairwise_all = best_configs_with_cola(metrics, rank)

  print("Concrete configs sorted by mean normalized score")
  display(config_summary_all.head(10))

  print("Friedman test across the top 4 concrete configs")
  display(config_friedman_all)

  print("Ordering by Friedman and mean score can be different")
  print("For significance claims we should use Friedman, because it is more consistency based, less prone to outliers, and therefore statistically more reliable")

  print("Wilcoxon post-hoc: top1 vs 2nd, 3rd, and 4th")
  display(config_pairwise_all)


These are the best concrete configs within the top 2 configured ranks
Top configured ranks: [2, 4]


 now we check within the best two ranks overll, which are specifically the best approaches within those
Configured rank 2: best concrete configs
Concrete configs sorted by mean normalized score


,configured_rank,run_id,kind,mean_score,n_rows
0,2,moelora_tr16_e8_k2,moelora,0.967458,42
1,2,hmora_tr16_e8_k2,hmora,0.940570,42
2,2,hydralora_r2,hydralora,0.939398,42
3,2,mola_tr16_e8_k6,mola,0.932036,42
4,2,mtllora_r2,mtllora,0.894854,42
5,2,lora_r2,lora,0.892762,42
6,2,cola_r2,cola,0.731217,42


Friedman test across the top 4 concrete configs


,configured_rank,top_configs_by_friedman_rank,n_pairs,friedman_p_raw,friedman_p_holm,significant,kendalls_w,avg_rank_moelora_tr16_e8_k2,avg_rank_mola_tr16_e8_k6,avg_rank_hydralora_r2,avg_rank_hmora_tr16_e8_k2
0,2,"[moelora_tr16_e8_k2, mola_tr16_e8_k6, hydralor...",42,0.000375,0.000375,True,0.145503,2.035714,2.297619,2.488095,3.178571


Ordering by Friedman and mean score can be different
For significance claims we should use Friedman, because it is more consistency based, less prone to outliers, and therefore statistically more reliable
Wilcoxon post-hoc: top1 vs 2nd, 3rd, and 4th


,a,b,mean_a,mean_b,median_delta_a_minus_b,n_pairs,p_raw,p_holm,significant
0,moelora_tr16_e8_k2,mola_tr16_e8_k6,0.967458,0.932036,0.000231,42,0.006880,0.013497,True
1,moelora_tr16_e8_k2,hydralora_r2,0.967458,0.939398,0.000285,42,0.006418,0.013497,True
2,moelora_tr16_e8_k2,hmora_tr16_e8_k2,0.967458,0.940570,0.000972,42,0.004499,0.013497,True


... (2 outputs left out)

Friedman test across the top 4 concrete configs


,configured_rank,top_configs_by_friedman_rank,n_pairs,friedman_p_raw,friedman_p_holm,significant,kendalls_w,avg_rank_mola_tr32_e8_k6,avg_rank_moelora_tr32_e8_k2,avg_rank_lora_r4,avg_rank_hydralora_r4
0,4,"[mola_tr32_e8_k6, moelora_tr32_e8_k2, lora_r4,...",42,0.144744,0.144744,False,0.042857,2.25,2.321429,2.607143,2.821429


Ordering by Friedman and mean score can be different
For significance claims we should use Friedman, because it is more consistency based, less prone to outliers, and therefore statistically more reliable
Wilcoxon post-hoc: top1 vs 2nd, 3rd, and 4th


,a,b,mean_a,mean_b,median_delta_a_minus_b,n_pairs,p_raw,p_holm,significant
0,mola_tr32_e8_k6,moelora_tr32_e8_k2,0.927662,0.931373,-0.000032,42,0.687102,0.687102,False
1,mola_tr32_e8_k6,lora_r4,0.927662,0.918843,0.000152,42,0.060498,0.181494,False
2,mola_tr32_e8_k6,hydralora_r4,0.927662,0.915862,0.000063,42,0.149657,0.299314,False


## Answer to our questions:
### Which is the two best rank configurations across all approaches:
    - the best rank configurations are `2` and `4`
---
### Which specific approaches perform the best within these best configured ranks:
    - the best three approaches when the rank `2` is configured are: 1. `moelora_tr16_e8_k2` 2. `mola_tr16_e8_k6` 3. `hydralora_r2`
    - the best three approaches when the rank `4` is configured are: 1. `mola_tr32_e8_k6` 2. `moelora_tr32_e8_k2` 3. `lora_r4`